# Train tinyformer on a free GPU

Runtime → Change runtime type → **T4 GPU**, then run the cells in order. Replace `sweeney-007` with your GitHub username.

In [ ]:
!git clone https://github.com/sweeney-007/tinyformer.git
%cd tinyformer
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Prepare data
Python code (built from this machine's standard library) with two tokenizers.

In [ ]:
!python prepare.py --dataset python-stdlib --out data/pycode_bpe --tokenizer bpe --vocab-size 2048 --max-chars 10000000
!python prepare.py --input data/pycode_bpe/input.txt --out data/pycode_char --tokenizer char

## 2. Train one model

In [ ]:
!python train.py --data data/pycode_bpe --out out --preset small --max-iters 5000 --fused-attention

In [ ]:
!python sample.py --out out --prompt "def " --max-new-tokens 200 --num-samples 3

## 3. Run the experiment suites (3 seeds each)

In [ ]:
!python experiments.py --suite tokenizer --data data/pycode_bpe data/pycode_char --iters 3000 --seeds 1 2 3 -- --preset small --fused-attention
!python experiments.py --suite arch --data data/pycode_bpe --iters 3000 --seeds 1 2 3 -- --preset small --fused-attention

In [ ]:
from IPython.display import Image, Markdown, display
for s in ['tokenizer', 'arch']:
    display(Markdown(open(f'runs/{s}/results.md').read()))
    display(Image(f'runs/{s}/curves.png'))

## 4. Save results
Download `runs/*/results.md` and `curves.png` and commit them to `docs/results/`, or copy them to Google Drive:

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r runs /content/drive/MyDrive/tinyformer_runs